In [4]:
# EfficientNet-B0 - Data Preparation

import os
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Paths
PROJECT_ROOT = "/Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit"
DATASET_ROOT = os.path.join(PROJECT_ROOT, "data")

TRAIN_DIR = "/Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Training"
TEST_DIR = "/Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Testing"

# Configuration
IMG_SIZE = 224
BATCH_SIZE = 8
NUM_CLASSES = 4
NUM_WORKERS = 0

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("Training directory:", TRAIN_DIR)
print("Testing directory:", TEST_DIR)

PyTorch version: 2.8.0
MPS available: True
Training directory: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Training
Testing directory: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Testing


In [5]:
# Image preprocessing and augmentation

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(7),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load datasets
full_train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform
)

print("Classes:", full_train_dataset.classes)
print("Training images:", len(full_train_dataset))
print("Testing images:", len(test_dataset))


Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Training images: 5600
Testing images: 1600


In [3]:
import os

base = "/Users/mirshad/Desktop/SLIIT/brain-tumor"

for root, dirs, files in os.walk(base):
    if "Training" in dirs or "Testing" in dirs:
        print("Found:", root)
        if "Training" in dirs:
            print("  Training:", os.path.join(root, "Training"))
        if "Testing" in dirs:
            print("  Testing:", os.path.join(root, "Testing"))

Found: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive
  Training: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Training
  Testing: /Users/mirshad/Desktop/SLIIT/brain-tumor/brain-tumor-vit/archive/Testing


In [6]:
# Create stratified train/validation split

targets = np.array(full_train_dataset.targets)

train_indices, val_indices = train_test_split(
    np.arange(len(targets)),
    test_size=0.20,
    random_state=SEED,
    stratify=targets
)

# Separate transforms for training and validation
train_dataset_full = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transform
)

val_dataset_full = datasets.ImageFolder(
    TRAIN_DIR,
    transform=eval_transform
)

train_dataset = Subset(train_dataset_full, train_indices)
val_dataset = Subset(val_dataset_full, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Batch size:", BATCH_SIZE)

Train: 4480
Validation: 1120
Test: 1600
Batch size: 8


In [7]:
# EfficientNet-B0 model setup

import timm

# Device
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)

# Load pretrained EfficientNet-B0
model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=NUM_CLASSES
)

model = model.to(DEVICE)

# Freeze the backbone for Stage 1
for param in model.parameters():
    param.requires_grad = False

# Train only the final classifier
for param in model.classifier.parameters():
    param.requires_grad = True

# Parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Model: EfficientNet-B0")
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Classifier:", model.classifier)

Using device: mps


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Model: EfficientNet-B0
Total parameters: 4012672
Trainable parameters: 5124
Classifier: Linear(in_features=1280, out_features=4, bias=True)


In [8]:
# EfficientNet-B0 - Stage 1 Training
# Frozen backbone + trainable classifier

import time
import json
import os
import torch.nn as nn

# Training configuration
STAGE1_EPOCHS = 5
STAGE1_LR = 1e-3
STAGE1_WEIGHT_DECAY = 1e-4

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=STAGE1_LR,
    weight_decay=STAGE1_WEIGHT_DECAY
)

# Check trainable parameters
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Stage 1 configuration")
print("----------------------")
print("Epochs:", STAGE1_EPOCHS)
print("Learning rate:", STAGE1_LR)
print("Weight decay:", STAGE1_WEIGHT_DECAY)
print("Trainable parameters:", trainable_params)
print("Device:", DEVICE)

# Store training history
stage1_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_accuracy = -1.0
best_epoch = 0

# Save checkpoints outside the Git repository
CHECKPOINT_DIR = os.path.expanduser("~/brain-tumor-checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_CHECKPOINT_PATH = os.path.join(
    CHECKPOINT_DIR,
    "efficientnet_b0_stage1_best.pth"
)

stage1_start_time = time.perf_counter()

for epoch in range(STAGE1_EPOCHS):

    # -------------------------
    # Training
    # -------------------------
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    epoch_start_time = time.perf_counter()

    for images, labels in train_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        running_loss += loss.item() * batch_size

        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += batch_size

    train_loss = running_loss / total
    train_accuracy = correct / total

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.inference_mode():

        for images, labels in val_loader:

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)

            val_running_loss += loss.item() * batch_size

            predictions = torch.argmax(outputs, dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += batch_size

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    # Store history
    stage1_history["train_loss"].append(train_loss)
    stage1_history["train_accuracy"].append(train_accuracy)
    stage1_history["val_loss"].append(val_loss)
    stage1_history["val_accuracy"].append(val_accuracy)

    epoch_time = time.perf_counter() - epoch_start_time

    print(
        f"Epoch [{epoch + 1}/{STAGE1_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy * 100:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy * 100:.2f}% | "
        f"Time: {epoch_time / 60:.2f} min"
    )

    # Save best validation checkpoint
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy
        best_epoch = epoch + 1

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": best_epoch,
                "val_accuracy": best_val_accuracy,
                "stage1_history": stage1_history,
                "class_names": CLASS_NAMES,
                "num_classes": NUM_CLASSES,
                "img_size": IMG_SIZE
            },
            BEST_CHECKPOINT_PATH
        )

        print(
            f"  ✓ Best checkpoint saved "
            f"(Val Acc: {best_val_accuracy * 100:.2f}%)"
        )

stage1_total_time = time.perf_counter() - stage1_start_time

print("\nStage 1 complete")
print("----------------")
print(f"Best validation accuracy: {best_val_accuracy * 100:.2f}%")
print(f"Best epoch: {best_epoch}")
print(f"Total training time: {stage1_total_time / 60:.2f} minutes")
print(f"Best checkpoint: {BEST_CHECKPOINT_PATH}")

Stage 1 configuration
----------------------
Epochs: 5
Learning rate: 0.001
Weight decay: 0.0001
Trainable parameters: 5124
Device: mps
Epoch [1/5] | Train Loss: 1.4396 | Train Acc: 59.15% | Val Loss: 0.9259 | Val Acc: 71.34% | Time: 0.95 min
  ✓ Best checkpoint saved (Val Acc: 71.34%)
Epoch [2/5] | Train Loss: 0.7834 | Train Acc: 75.80% | Val Loss: 0.6848 | Val Acc: 78.48% | Time: 0.84 min
  ✓ Best checkpoint saved (Val Acc: 78.48%)
Epoch [3/5] | Train Loss: 0.7139 | Train Acc: 77.46% | Val Loss: 0.6423 | Val Acc: 79.46% | Time: 0.83 min
  ✓ Best checkpoint saved (Val Acc: 79.46%)
Epoch [4/5] | Train Loss: 0.5875 | Train Acc: 80.36% | Val Loss: 0.5460 | Val Acc: 82.41% | Time: 0.84 min
  ✓ Best checkpoint saved (Val Acc: 82.41%)
Epoch [5/5] | Train Loss: 0.5692 | Train Acc: 81.34% | Val Loss: 0.5043 | Val Acc: 82.95% | Time: 0.83 min
  ✓ Best checkpoint saved (Val Acc: 82.95%)

Stage 1 complete
----------------
Best validation accuracy: 82.95%
Best epoch: 5
Total training time: 4.30 m